# House Price Prediction using Machine Learning

**Author:** _your name_  
**Dataset:** California Housing (`sklearn.datasets.fetch_california_housing`)  
**Task:** Regression — predict median house value (`MedHouseVal`, units of $100,000)

This notebook is the hiring-manager path: load, look, clean without leaking the test fold, train four models, and read the winner. Heavy lifting lives in `src/` so the cells stay short.

**Holdout result:** XGBoost, R² **0.844**, RMSE **$45,261**, MAE **$29,923**.


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import FEATURE_COLUMNS, TARGET, load_housing_frame, missing_value_report, preprocess
from src.models import feature_importance_frame, fit_and_score, scores_to_frame
from src.visualize import (
    plot_actual_vs_predicted,
    plot_correlation_heatmap,
    plot_feature_importance,
    plot_geography,
    plot_income_vs_price,
    plot_model_comparison,
    plot_residuals,
    plot_target_distribution,
)

pd.set_option("display.precision", 3)
%matplotlib inline


## 2. Load and explore

In [ ]:
frame = load_housing_frame()
print("shape:", frame.shape)
print("\ndtypes:\n", frame.dtypes)
print("\nmissing values:\n", missing_value_report(frame))
frame.head()


In [ ]:
frame.describe().T

California Housing is already numeric and complete (20,640 × 9 including the target). `MedHouseVal` is censored at 5.00001 — $500,000 — which will show up later as a wall in the residual plots.


In [ ]:
plot_target_distribution(frame[TARGET], ROOT / "artifacts" / "01_target_distribution.png")
plt.show()


The right-hand spike is the $500k cap, not a cluster of identical homes.

In [ ]:
plot_correlation_heatmap(frame, ROOT / "artifacts" / "02_correlation_heatmap.png")
plt.show()


In [ ]:
plot_income_vs_price(frame, ROOT / "artifacts" / "03_income_vs_price.png")
plt.show()


Median income is the strongest linear correlate of value. Geography is the other half of the story.

In [ ]:
plot_geography(frame, ROOT / "artifacts" / "04_geography.png")
plt.show()


## 3. Preprocessing and split

Winsorize occupancy and room ratios using **training** 1st/99th percentiles, then fit `StandardScaler` on train only. Trees consume the unscaled matrix; linear regression consumes the scaled copy.


In [ ]:
split = preprocess(frame)
print(f"train={len(split.X_train)}  test={len(split.X_test)}")
print("scaler means (train):")
pd.Series(split.scaler.mean_, index=FEATURE_COLUMNS)


## 4. Train and compare

In [ ]:
scores = fit_and_score(split)
table = scores_to_frame(scores)
table

| Model | MAE | RMSE | R² |
| --- | ---: | ---: | ---: |
| **XGBoost** | **0.299** | **0.453** | **0.844** |
| Random Forest | 0.327 | 0.505 | 0.805 |
| Decision Tree | 0.405 | 0.598 | 0.727 |
| Linear Regression | 0.498 | 0.680 | 0.647 |

XGBoost is the holdout winner. Boosted trees capture coastal / inland breaks that a linear plane cannot.


In [ ]:
plot_model_comparison(table, ROOT / "artifacts" / "05_model_comparison.png")
plt.show()


## 5. Best model — XGBoost

In [ ]:
winner = scores[0]
print(winner.name, "RMSE", round(winner.rmse, 4), "R2", round(winner.r2, 4))
importance = feature_importance_frame(winner.estimator, list(FEATURE_COLUMNS))
importance


In [ ]:
plot_feature_importance(importance, ROOT / "artifacts" / "08_feature_importance.png")
plt.show()


- **MedInc (37.8%)** — ability to pay.
- **AveOccup (13.8%)** — crowding discounts the block.
- **Longitude + Latitude (~26%)** — location, mostly coast versus interior.
- **Rooms > bedrooms > population.**


In [ ]:
plot_actual_vs_predicted(split.y_test.to_numpy(), winner.y_pred, ROOT / "artifacts" / "06_actual_vs_predicted.png")
plt.show()


In [ ]:
plot_residuals(split.y_test.to_numpy(), winner.y_pred, ROOT / "artifacts" / "07_residuals.png")
plt.show()


Residuals stay centered until predictions hit the $500k label cap, where the target itself is censored.

## 6. Persist the winner

In [ ]:
import joblib

joblib.dump(winner.estimator, ROOT / "artifacts" / "best_model.joblib")
joblib.dump(split.scaler, ROOT / "artifacts" / "preprocessor.joblib")
table.to_csv(ROOT / "reports" / "model_comparison.csv", index=False)
print("saved", ROOT / "artifacts" / "best_model.joblib")


## 7. Conclusion

A regularized linear model is a respectable baseline (R² 0.65) but leaves about $68k of RMSE on the table. A depth-tuned tree improves that; bagging and especially boosting close most of the remaining gap. For production scoring I would ship the XGBoost booster and keep the linear model around as an explainer.

**Next steps:** Ames Housing (categoricals), Optuna on the booster, spatial cross-validation, SHAP.
